# Long-Term Memory (the Store)

**What you will build:** an agent that remembers facts about a **user across different conversations** — not just within one thread. In 2.3 the checkpointer gave each `thread_id` its own memory; that's *short-term*. LangGraph's **Store** is *long-term*: it remembers a user even in a brand-new thread. This is the memory tier most "personal assistant" features actually need, and it's new versus everything before it.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/28_long_term_memory.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)

## Two tiers of memory

```
Checkpointer (2.3)   keyed by thread_id    remembers ONE conversation      short-term
Store (this)         keyed by user_id      remembers a USER across threads  long-term
```

You attach **both** when you compile: the checkpointer for the running conversation, the store for durable, cross-thread facts. A node reads and writes the store via the injected `runtime.store`.

In [ ]:
import uuid
from dataclasses import dataclass
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime

@dataclass
class Context:
    user_id: str          # who we're talking to — the key for long-term memory

def call_model(state: MessagesState, runtime: Runtime[Context]) -> dict:
    ns = ("memories", runtime.context.user_id)          # this user's namespace
    known = [m.value["data"] for m in runtime.store.search(ns)]   # everything we remember about them

    last = state["messages"][-1].content
    if "remember" in last.lower():                       # naive: store anything they ask us to remember
        runtime.store.put(ns, str(uuid.uuid4()), {"data": last})

    system = "What you already know about the user:\n" + ("\n".join(known) or "(nothing yet)")
    resp = llm.invoke([{"role": "system", "content": system}, *state["messages"]])
    return {"messages": resp}

builder = StateGraph(MessagesState, context_schema=Context)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

graph = builder.compile(checkpointer=InMemorySaver(), store=InMemoryStore())

## Prove it: remember in one thread, recall in another

Tell the agent a fact in conversation 1, then start a **brand-new** conversation (different `thread_id`) for the same user and see it recall the fact.

In [ ]:
user = Context(user_id="u-42")

# Conversation 1
graph.invoke({"messages": [{"role": "user", "content": "Please remember: my name is Bob and I love hiking."}]},
             config={"configurable": {"thread_id": "conv-1"}}, context=user)

# Conversation 2 — a DIFFERENT thread, no shared checkpointer state
r = graph.invoke({"messages": [{"role": "user", "content": "What's my name, and what do I like to do?"}]},
                 config={"configurable": {"thread_id": "conv-2"}}, context=user)
print(r["messages"][-1].content)   # recalls Bob + hiking, across threads

It remembered — even though conversation 2 is a fresh thread with no checkpointer history. The fact lived in the **Store**, keyed by `user_id`, not by thread. Switch to a different `user_id` and the memory is gone: that's how one app serves many users without their memories bleeding together.

Checkpointer = *this conversation*; store = *this person*. Real assistants need both.

::::{dropdown} 🔍 Semantic long-term memory
:color: info

Here we retrieved **all** of a user's memories with `store.search(ns)`. For many memories you want the *relevant* ones — that's RAG (1.7) applied to memory. Give the store an embedding index and `search(ns, query=...)` returns the closest matches:

```python
from langchain.embeddings import init_embeddings
store = InMemoryStore(index={"embed": init_embeddings("openai:text-embedding-3-small"),
                             "dims": 1536, "fields": ["data"]})
# ... then: runtime.store.search(ns, query=last, limit=3)
```

`embed` also accepts a plain function, so you can plug in the keyless `sentence-transformers` embedder from 1.7 instead of a paid embeddings API.
::::

::::{dropdown} 🔧 Common issues (and fixes)
:color: secondary

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| **Doesn't recall across threads** | Only a checkpointer, no store | Compile with `store=...` and read it via `runtime.store` |
| **Users see each other's memories** | Shared namespace | Namespace by `user_id`: `("memories", user_id)` |
| **`runtime.store` is None** | Compiled without a store, or wrong node signature | Pass `store=` at compile; type the param `Runtime[Context]` |
::::

**What's next — Block 3:** you have all the pieces. Now we ship them — deploy an agent behind a real **API** (30), connect a **front end** (31), and build a **capstone project** (32).